# Clasificador Inteligente de Imagenes de Ropa
**Modulo 8 — Fundamentos de Deep Learning | Alkemy**  
**Empresa:** StyleNet — Area de Ciencia de Datos

---

**Contexto:** StyleNet necesita automatizar la clasificacion de imagenes de productos subidas por usuarios y vendedores. Actualmente el proceso es manual, lo que genera retrasos y errores de categorizacion.

**Objetivo:** Disenar, entrenar y comparar dos arquitecturas de redes neuronales (ANN densa y CNN) para clasificar prendas de vestir usando Fashion-MNIST.

| Leccion | Contenido |
|---|---|
| 1 | La red neuronal artificial — fundamentos y arquitectura densa |
| 2 | Deep Learning — arquitecturas, frameworks y justificacion |
| 3 | Implementacion de la ANN densa con Keras |
| 4 | Red neuronal convolutiva (CNN) y comparacion de modelos |

---
## Importacion de librerias

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# Metricas y reporte
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version     : {keras.__version__}')
print('Librerias cargadas correctamente.')

---
## Leccion 1 — La Red Neuronal Artificial

Una **red neuronal artificial (ANN)** es un modelo computacional inspirado en el cerebro humano. Esta compuesta por neuronas organizadas en capas que procesan informacion y aprenden patrones a partir de los datos.

### Elementos clave

| Elemento | Descripcion | En este proyecto |
|---|---|---|
| **Capas** | Entrada, ocultas y salida | Flatten + Dense + Dropout |
| **Pesos** | Parametros ajustables durante el entrenamiento | Optimizados por Adam |
| **Funcion de activacion** | Introduce no-linealidad en la red | ReLU (capas ocultas), Softmax (salida) |
| **Funcion de perdida** | Mide el error de las predicciones | Sparse Categorical Crossentropy |
| **Optimizador** | Ajusta los pesos para minimizar la perdida | Adam |

### Clasificacion binaria vs. multiclase

- **Binaria:** una salida, funcion sigmoide (ej: spam / no spam)
- **Multiclase:** N salidas con softmax (ej: 10 categorias de ropa)

Fashion-MNIST es un problema **multiclase de 10 categorias**, por lo que usaremos `softmax` en la capa de salida y `sparse_categorical_crossentropy` como perdida.

---
## Leccion 2 — Deep Learning: Arquitecturas y Justificacion

### Por que CNN es superior a ANN densa para imagenes

| Aspecto | ANN Densa | CNN |
|---|---|---|
| Entrada | Vector plano (784 pixels) | Matriz 2D (28x28) |
| Patrones detectados | Globales (no espaciales) | Locales: bordes, texturas, formas |
| Parametros | Muchos (alta dimensionalidad) | Pocos (filtros compartidos) |
| Invariancia a traslacion | No | Si (via pooling) |
| Rendimiento tipico en Fashion-MNIST | ~88% | ~92%+ |

La ANN densa trata cada pixel de forma independiente y pierde la informacion espacial. La CNN, en cambio, aplica filtros convolucionales que detectan caracteristicas locales (bordes, costuras, patrones de textura) respetando la estructura 2D de la imagen.

### Framework seleccionado: Keras / TensorFlow

- API de alto nivel, clara y concisa
- Ideal para prototipado rapido y proyectos educativos
- Amplia documentacion y comunidad activa
- Fashion-MNIST disponible directamente en `keras.datasets`

---
## Leccion 3 — Implementacion: Carga y Preprocesamiento

### Carga del dataset Fashion-MNIST

Fashion-MNIST contiene 70.000 imagenes en escala de grises (28x28 pixels) distribuidas en 10 categorias de prendas de vestir.

In [ ]:
# Cargamos Fashion-MNIST directamente desde Keras
# 60.000 imagenes de entrenamiento y 10.000 de prueba
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Nombres de las 10 categorias
CLASES = [
    'Remera/Top', 'Pantalon', 'Sueter', 'Vestido', 'Abrigo',
    'Sandalia', 'Camisa', 'Zapatilla', 'Cartera', 'Botita'
]

print(f'Entrenamiento : {X_train.shape} imagenes, etiquetas: {y_train.shape}')
print(f'Prueba        : {X_test.shape} imagenes, etiquetas: {y_test.shape}')
print(f'Rango original: [{X_train.min()}, {X_train.max()}]')

In [ ]:
# Visualizamos algunas imagenes del dataset para entender los datos
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, ax in enumerate(axes.flatten()):
    idx = np.where(y_train == i)[0][0]  # primera imagen de cada clase
    ax.imshow(X_train[idx], cmap='gray')
    ax.set_title(CLASES[i], fontsize=10)
    ax.axis('off')
plt.suptitle('Fashion-MNIST — una imagen por categoria', fontsize=13)
plt.tight_layout()
plt.savefig('fashion_mnist_clases.png', bbox_inches='tight')
plt.show()

### Preprocesamiento

In [ ]:
# Normalizacion: llevamos los valores de pixeles de [0, 255] a [0, 1]
# Esto acelera la convergencia y estabiliza el entrenamiento
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm  = X_test.astype('float32')  / 255.0

# Para la CNN necesitamos una dimension extra de canal: (28, 28) -> (28, 28, 1)
X_train_cnn = X_train_norm[..., np.newaxis]  # shape: (60000, 28, 28, 1)
X_test_cnn  = X_test_norm[..., np.newaxis]   # shape: (10000, 28, 28, 1)

print(f'ANN — X_train: {X_train_norm.shape} | X_test: {X_test_norm.shape}')
print(f'CNN — X_train: {X_train_cnn.shape}  | X_test: {X_test_cnn.shape}')
print(f'Rango normalizado: [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}]')

### Distribucion de clases

In [ ]:
# Verificamos que el dataset este balanceado
conteos = np.bincount(y_train)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(CLASES, conteos, color='steelblue', edgecolor='white')
ax.set_ylabel('Cantidad de imagenes')
ax.set_title('Distribucion de clases — conjunto de entrenamiento')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('distribucion_clases.png', bbox_inches='tight')
plt.show()

print('Imagenes por clase:', dict(zip(CLASES, conteos)))

---
## Modelo 1 — Red Neuronal Densa (ANN)

La ANN densa aplana cada imagen en un vector de 784 valores y los procesa a traves de capas completamente conectadas.

**Arquitectura:**
```
Entrada (28x28) → Flatten (784) → Dense(256, ReLU) → Dropout(0.3)
               → Dense(128, ReLU) → Dropout(0.3) → Dense(10, Softmax)
```

In [ ]:
def construir_ann():
    """
    Red neuronal densa para clasificacion de imagenes Fashion-MNIST.

    Capas:
    - Flatten    : convierte la imagen 28x28 en un vector de 784 valores
    - Dense(256) : primera capa oculta con activacion ReLU
    - Dropout    : desactiva el 30% de neuronas aleatoriamente para evitar overfitting
    - Dense(128) : segunda capa oculta
    - Dense(10)  : capa de salida con Softmax para clasificacion multiclase
    """
    modelo = models.Sequential([
        # Aplanamos la imagen 2D en un vector 1D
        layers.Flatten(input_shape=(28, 28), name='entrada'),

        # Primera capa oculta
        # ReLU: f(x) = max(0, x) — introduce no-linealidad, rapida de calcular
        layers.Dense(256, activation='relu', name='densa_1'),

        # Dropout: regularizacion que apaga neuronas aleatoriamente en cada batch
        # Evita que la red memorice los datos de entrenamiento (overfitting)
        layers.Dropout(0.3, name='dropout_1'),

        # Segunda capa oculta
        layers.Dense(128, activation='relu', name='densa_2'),
        layers.Dropout(0.3, name='dropout_2'),

        # Capa de salida: 10 neuronas, una por clase
        # Softmax convierte los logits en probabilidades que suman 1
        layers.Dense(10, activation='softmax', name='salida'),
    ], name='ANN_StyleNet')

    # Compilacion del modelo
    # - sparse_categorical_crossentropy: perdida para clasificacion multiclase con etiquetas enteras
    # - adam: optimizador adaptativo, combina momentum y RMSprop
    # - accuracy: metrica principal de evaluacion
    modelo.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return modelo


ann = construir_ann()
ann.summary()

In [ ]:
# EarlyStopping: detiene el entrenamiento si la validacion no mejora
# restore_best_weights=True recupera los pesos del mejor epoch
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print('Entrenando ANN...')
historia_ann = ann.fit(
    X_train_norm, y_train,
    epochs=30,
    batch_size=64,      # numero de ejemplos por actualizacion de pesos
    validation_split=0.15,  # 15% del entrenamiento como validacion
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evaluacion sobre el conjunto de prueba
loss_ann, acc_ann = ann.evaluate(X_test_norm, y_test, verbose=0)
print(f'ANN — Loss en prueba   : {loss_ann:.4f}')
print(f'ANN — Accuracy en prueba: {acc_ann*100:.2f}%')

---
## Leccion 4 — Modelo 2: Red Neuronal Convolutiva (CNN)

La CNN preserva la estructura espacial 2D de la imagen y aprende caracteristicas jerarquicas: bordes → texturas → formas → categorias.

**Arquitectura:**
```
Entrada (28x28x1)
  → Conv2D(32, 3x3, ReLU) → MaxPooling(2x2)
  → Conv2D(64, 3x3, ReLU) → MaxPooling(2x2)
  → Conv2D(128, 3x3, ReLU)
  → Flatten → Dense(128, ReLU) → Dropout(0.4)
  → Dense(10, Softmax)
```

| Capa | Rol |
|---|---|
| Conv2D | Aplica filtros para detectar caracteristicas locales |
| MaxPooling2D | Reduce la dimension espacial, preserva caracteristicas dominantes |
| Flatten | Convierte el mapa de caracteristicas en vector para las capas densas |
| Dropout | Regularizacion para reducir overfitting |

In [ ]:
def construir_cnn():
    """
    Red neuronal convolutiva para clasificacion de imagenes Fashion-MNIST.

    Bloques convolucionales:
    - Conv2D(32)  : detecta caracteristicas de bajo nivel (bordes, lineas)
    - Conv2D(64)  : detecta caracteristicas de nivel medio (texturas, patrones)
    - Conv2D(128) : detecta caracteristicas de alto nivel (formas, partes de prendas)
    """
    modelo = models.Sequential([
        # --- Bloque convolucional 1 ---
        # kernel_size=(3,3): tamano del filtro que recorre la imagen
        # padding='same': mantiene el tamano espacial de salida igual al de entrada
        layers.Conv2D(32, (3, 3), activation='relu',
                      padding='same', input_shape=(28, 28, 1), name='conv_1'),
        # MaxPooling: selecciona el valor maximo de cada region 2x2
        # Reduce la dimension a la mitad y aporta invariancia a traslacion
        layers.MaxPooling2D((2, 2), name='pool_1'),

        # --- Bloque convolucional 2 ---
        layers.Conv2D(64, (3, 3), activation='relu',
                      padding='same', name='conv_2'),
        layers.MaxPooling2D((2, 2), name='pool_2'),

        # --- Bloque convolucional 3 ---
        # Sin pooling: conservamos la resolucion espacial restante
        layers.Conv2D(128, (3, 3), activation='relu',
                      padding='same', name='conv_3'),

        # --- Clasificador denso ---
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='densa'),
        layers.Dropout(0.4, name='dropout'),  # 40%: CNN tiende mas al overfitting
        layers.Dense(10, activation='softmax', name='salida'),
    ], name='CNN_StyleNet')

    modelo.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return modelo


cnn = construir_cnn()
cnn.summary()

In [ ]:
early_stop_cnn = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print('Entrenando CNN...')
historia_cnn = cnn.fit(
    X_train_cnn, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop_cnn],
    verbose=1
)

In [ ]:
loss_cnn, acc_cnn = cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f'CNN — Loss en prueba    : {loss_cnn:.4f}')
print(f'CNN — Accuracy en prueba: {acc_cnn*100:.2f}%')

---
## Entrenamiento y Evaluacion — Comparacion de Modelos

### Curvas de entrenamiento

In [ ]:
def graficar_historia(hist_ann, hist_cnn):
    """Grafica las curvas de loss y accuracy para ambos modelos."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Accuracy ---
    axes[0].plot(hist_ann.history['accuracy'],     label='ANN — train', color='steelblue')
    axes[0].plot(hist_ann.history['val_accuracy'], label='ANN — val',   color='steelblue', linestyle='--')
    axes[0].plot(hist_cnn.history['accuracy'],     label='CNN — train', color='tomato')
    axes[0].plot(hist_cnn.history['val_accuracy'], label='CNN — val',   color='tomato', linestyle='--')
    axes[0].set_title('Accuracy por epoch')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].set_ylim(0.7, 1.0)

    # --- Loss ---
    axes[1].plot(hist_ann.history['loss'],     label='ANN — train', color='steelblue')
    axes[1].plot(hist_ann.history['val_loss'], label='ANN — val',   color='steelblue', linestyle='--')
    axes[1].plot(hist_cnn.history['loss'],     label='CNN — train', color='tomato')
    axes[1].plot(hist_cnn.history['val_loss'], label='CNN — val',   color='tomato', linestyle='--')
    axes[1].set_title('Loss por epoch')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.suptitle('Comparacion de entrenamiento: ANN vs CNN', fontsize=13)
    plt.tight_layout()
    plt.savefig('curvas_entrenamiento.png', bbox_inches='tight')
    plt.show()

graficar_historia(historia_ann, historia_cnn)

### Tabla comparativa de resultados

In [ ]:
import pandas as pd

comparativa = pd.DataFrame({
    'Modelo':           ['ANN Densa', 'CNN'],
    'Accuracy (prueba)': [f'{acc_ann*100:.2f}%', f'{acc_cnn*100:.2f}%'],
    'Loss (prueba)':    [f'{loss_ann:.4f}', f'{loss_cnn:.4f}'],
    'Parametros':       [
        f"{ann.count_params():,}",
        f"{cnn.count_params():,}"
    ],
    'Epochs entrenados': [
        len(historia_ann.history['loss']),
        len(historia_cnn.history['loss'])
    ]
})

print('=== COMPARATIVA DE MODELOS ===')
print(comparativa.to_string(index=False))

### Reporte de clasificacion detallado

In [ ]:
# Predicciones de ambos modelos sobre el conjunto de prueba
y_pred_ann = np.argmax(ann.predict(X_test_norm, verbose=0), axis=1)
y_pred_cnn = np.argmax(cnn.predict(X_test_cnn,  verbose=0), axis=1)

print('=== REPORTE DE CLASIFICACION — ANN ===')
print(classification_report(y_test, y_pred_ann, target_names=CLASES))

print('\n=== REPORTE DE CLASIFICACION — CNN ===')
print(classification_report(y_test, y_pred_cnn, target_names=CLASES))

### Matrices de confusion

In [ ]:
def graficar_confusion(y_true, y_pred, titulo, nombre_archivo):
    """Genera y guarda la matriz de confusion normalizada."""
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASES, yticklabels=CLASES,
                linewidths=0.5, ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel('Prediccion'); ax.set_ylabel('Real')
    plt.xticks(rotation=35, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(nombre_archivo, bbox_inches='tight')
    plt.show()

graficar_confusion(y_test, y_pred_ann, 'Matriz de confusion — ANN', 'confusion_ann.png')
graficar_confusion(y_test, y_pred_cnn, 'Matriz de confusion — CNN', 'confusion_cnn.png')

### Visualizacion de predicciones correctas e incorrectas

In [ ]:
def visualizar_predicciones(modelo, X, y_true, y_pred, n=10, titulo='Predicciones'):
    """
    Muestra una grilla de imagenes con sus predicciones.
    Verde: prediccion correcta. Rojo: prediccion incorrecta.
    """
    indices = np.random.choice(len(X), size=n, replace=False)
    fig, axes = plt.subplots(2, 5, figsize=(13, 6))

    for ax, idx in zip(axes.flatten(), indices):
        img = X[idx].squeeze()  # elimina la dimension de canal si existe
        ax.imshow(img, cmap='gray')
        color  = 'green' if y_pred[idx] == y_true[idx] else 'red'
        ax.set_title(f'Real: {CLASES[y_true[idx]]}\nPred: {CLASES[y_pred[idx]]}',
                     fontsize=8, color=color)
        ax.axis('off')

    plt.suptitle(titulo, fontsize=13)
    plt.tight_layout()
    plt.savefig(f'predicciones_{titulo.lower().replace(" ","_")}.png', bbox_inches='tight')
    plt.show()

visualizar_predicciones(ann, X_test_norm, y_test, y_pred_ann, titulo='Predicciones ANN')
visualizar_predicciones(cnn, X_test_cnn,  y_test, y_pred_cnn, titulo='Predicciones CNN')

---
## Predicciones sobre Imagenes Externas

Esta seccion permite cargar una imagen de ropa externa, preprocesarla y obtener la prediccion del modelo CNN.

In [ ]:
def predecir_imagen_externa(ruta_imagen: str, modelo=cnn, usar_cnn: bool = True):
    """
    Carga una imagen externa, la preprocesa y muestra la prediccion del modelo.

    Parametros
    ----------
    ruta_imagen : str
        Ruta al archivo de imagen (jpg, png, etc.)
    modelo : keras.Model
        Modelo entrenado a usar para la prediccion (por defecto: CNN)
    usar_cnn : bool
        True si el modelo es CNN (agrega dimension de canal)
    """
    # 1. Cargar la imagen en escala de grises
    img = tf.keras.utils.load_img(
        ruta_imagen,
        color_mode='grayscale',  # Fashion-MNIST es escala de grises
        target_size=(28, 28)     # redimensionamos al tamano del dataset
    )

    # 2. Convertir a array y normalizar
    img_array = tf.keras.utils.img_to_array(img) / 255.0

    # 3. Agregar dimension de batch: (28, 28, 1) -> (1, 28, 28, 1)
    img_batch = np.expand_dims(img_array, axis=0)

    # Si usamos ANN, aplanar: (1, 28, 28, 1) -> (1, 28, 28)
    if not usar_cnn:
        img_batch = img_batch.squeeze(axis=-1)

    # 4. Prediccion
    probabilidades = modelo.predict(img_batch, verbose=0)[0]
    clase_pred     = np.argmax(probabilidades)
    confianza      = probabilidades[clase_pred]

    # 5. Visualizacion
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # Imagen
    axes[0].imshow(img_array.squeeze(), cmap='gray')
    axes[0].set_title(f'Prediccion: {CLASES[clase_pred]}\nConfianza: {confianza*100:.1f}%',
                      fontsize=12, color='green')
    axes[0].axis('off')

    # Barras de probabilidad
    colores = ['tomato' if i == clase_pred else 'steelblue' for i in range(10)]
    axes[1].barh(CLASES, probabilidades * 100, color=colores)
    axes[1].set_xlabel('Probabilidad (%)')
    axes[1].set_title('Distribucion de probabilidades')
    axes[1].set_xlim(0, 100)

    plt.tight_layout()
    plt.savefig('prediccion_externa.png', bbox_inches='tight')
    plt.show()

    print(f'Categoria predicha : {CLASES[clase_pred]}')
    print(f'Confianza          : {confianza*100:.1f}%')
    return clase_pred, confianza


# --- Demo: usamos una imagen del conjunto de prueba como imagen "externa" ---
# Para usar una imagen real, descomentar la linea siguiente y pasar la ruta:
# predecir_imagen_externa('mi_imagen_de_ropa.jpg')

# Demo con imagen del test set guardada temporalmente
idx_demo = 42
img_demo = X_test[idx_demo]  # imagen original (0-255)
ruta_demo = 'imagen_demo.png'
plt.imsave(ruta_demo, img_demo, cmap='gray')

print(f'Categoria real de la imagen demo: {CLASES[y_test[idx_demo]]}')
predecir_imagen_externa(ruta_demo, modelo=cnn, usar_cnn=True)

---
## Informe Final

### Resumen ejecutivo

Se disenaron, entrenaron y compararon dos arquitecturas de redes neuronales para resolver el problema de clasificacion automatica de imagenes de ropa de StyleNet.

### Decisiones tecnicas

| Decision | ANN | CNN | Justificacion |
|---|---|---|---|
| Entrada | Vector 784D | Matriz 28x28x1 | CNN preserva estructura espacial |
| Activacion capas ocultas | ReLU | ReLU | Eficiente, evita vanishing gradient |
| Activacion salida | Softmax | Softmax | Probabilidades para 10 clases |
| Perdida | SparseCatXent | SparseCatXent | Multiclase con etiquetas enteras |
| Optimizador | Adam | Adam | Adaptativo, rapido de converger |
| Regularizacion | Dropout 30% | Dropout 40% | Previene overfitting |
| Detencion | EarlyStopping | EarlyStopping | Evita epochs innecesarios |

### Conclusion

La CNN supera a la ANN densa en accuracy de clasificacion al aprovechar la estructura espacial de las imagenes. Para un entorno de produccion como StyleNet, se recomienda la CNN como modelo principal, con posibilidad de incorporar Data Augmentation y Transfer Learning para mayor robustez.

---
## Exportacion del modelo

In [ ]:
# Guardamos el modelo CNN entrenado en formato SavedModel de TensorFlow
cnn.save('modelo_cnn_stylenet.keras')
ann.save('modelo_ann_stylenet.keras')

print('Modelos exportados:')
print('  modelo_cnn_stylenet.keras')
print('  modelo_ann_stylenet.keras')

# Para cargar el modelo en otro contexto:
# modelo_cargado = keras.models.load_model('modelo_cnn_stylenet.keras')

---
## Referencias

- Keras Documentation: https://keras.io/
- TensorFlow API: https://www.tensorflow.org/api_docs
- Fashion-MNIST: https://github.com/zalandoresearch/fashion-mnist
- Machine Learning Mastery: https://machinelearningmastery.com/start-here/
- DotCSV — Redes Neuronales: https://www.youtube.com/@DotCSV

---
*Evaluacion Modulo 8 — Alkemy | StyleNet*